In [ ]:
from random import randint

: 

In [2]:
def initialize_board():
    board=list()
    for _ in range(3):
        board.append(['-', '-', '-'])
    return board

In [3]:
def setup_board(board, random_count):
    current_state=[row[:] for row in board]
    while random_count:
        row, column=randint(0, 2), randint(0, 2)
        if current_state[row][column]=='-':
            # considering first move made by 'x'
            current_state[row][column]='o' if random_count%2==0 else 'x'
            random_count-=1
    return current_state

In [4]:
def display_board(board):
    for _ in range(3):
        print(board[_])

In [5]:
def check_win(board):
    for _ in range(3):
        if board[_][0]==board[_][1]==board[_][2] and board[_][0]!='-':
            return board[_][0]
        if board[0][_]==board[1][_]==board[2][_] and board[0][_]!='-':
            return board[0][_]
    # check for diagonals
    if board[0][0]==board[1][1]==board[2][2] and board[0][0]!='-':
        return board[0][0]
    if board[0][2]==board[1][1]==board[2][0] and board[0][2]!='-':
        return board[0][2]
    return None

In [6]:
def check_draw(board):
    # it's only if someone accidentally checks for draw on a winning position
    # it's more of like error handling
    if check_win(board) is not None:
        return False
    for i in range(3):
        for j in range(3):
            if board[i][j]=='-':
                return False
    # there is no move available on board and someone has not won yet
    return True

In [7]:
def get_player_turn(board):
    count_x, count_o=0, 0
    for i in range(3):
        for j in range(3):
            if board[i][j]=='x':
                count_x+=1
            elif board[i][j]=='o':
                count_o+=1
    # first move by x
    return 'x' if count_x<=count_o else 'o'

In [8]:
def get_moves(board):
    moves=tuple()
    for i in range(3):
        for j in range(3):
            if board[i][j]=='-':
                moves=moves+((i, j),)
    return moves

In [9]:
def get_children_values(children, states_points):
    values=list()
    for child in children:
        values.append(get_state_value(board=child, states_points=states_points))
    return values

In [10]:
def get_child_states(board):
    possible_moves=get_moves(board=board)
    current_player=get_player_turn(board=board)
    children=list()
    for _ in range(len(possible_moves)):
        # deep copy current state
        current_state=[row[:] for row in board]
        current_state[int(possible_moves[_][0])][int(possible_moves[_][1])]=current_player
        children.append(current_state)
    return children

In [11]:
def get_state_value(board, states_points):
    current_state_tuple=board.copy()
    current_state_tuple=tuple(tuple(row) for row in current_state_tuple)
    current_player=get_player_turn(board=board)
    # base case, it will only be applied at leaf node
    if current_state_tuple in states_points.keys():
        return states_points[current_state_tuple]
    if check_win(board=board) is not None:
        # at 'o' turn it see that 'x' has already won the game
        states_points[current_state_tuple]=1 if current_player=='o' else -1
        return states_points[current_state_tuple]
    if check_draw(board=board):
        states_points[current_state_tuple]=0
        return 0

    # recursive case, it will be applied at all non leaf nodes taking values from children using minimax concept
    # take all the possible moves from the given state
    child_states=get_child_states(board=board)
    child_values=get_children_values(children=child_states, states_points=states_points)
    states_points[current_state_tuple]=min(child_values) if current_player=='o' else max(child_values)
    return states_points[current_state_tuple]

In [12]:
def get_next_child(current_state, states_points):
    if check_draw(board=current_state) or check_win(board=current_state):
        return current_state
    children=get_child_states(board=current_state)
    children_values=get_children_values(children=children, states_points=states_points)
    current_player=get_player_turn(board=current_state)
    if current_player=='x':
        required_value=max(children_values)
    else:
        required_value=min(children_values)
    for key, value in states_points.items():
        if value == required_value:
            target_key=[list(row) for row in key]
            if target_key in children:
                return key
    return None


In [13]:
def minimax(board):
    state_points=dict()
    board_result=get_state_value(board=board, states_points=state_points)
    if board_result==0:
        print('Match will draw')
    else:
        print('X has more chances of winning') if board_result==1 else print('O has more chances of winning')

In [14]:
def get_states_values(board):
    states_points=dict()
    get_state_value(board=board, states_points=states_points)
    return states_points

In [15]:
def get_computer_next_move(board, states_points, children, child_values):
    max_value=max(child_values)
    for key, value in states_points.items():
        if value == max_value:
            target_key=[list(row) for row in key]
            if target_key in children:
                # have to check if computer made 'x' move
                return list(list(row[:]) for row in key)
    # None show there is some error and so for error handing check None
    return None

In [16]:
def computer_turn(board, states_points):
    # it has to maximize the result every time
    current_state=list([list(row[:]) for row in board])
    children=get_child_states(board=current_state)
    child_values=get_children_values(children=children, states_points=states_points)
    return get_computer_next_move(board=current_state, states_points=states_points, children=children, child_values=child_values)

In [17]:
def human_turn(board):
    current_state=list([list(row[:]) for row in board])
    while True:
        print('Enter a valid cell to place your mark!')
        try:
            row=int(input('Row: '))
            column=int(input('Column: '))
        except ValueError:
            print('Enter integer values')
            continue
        if not (1<=row,column<=3):
            print('Rows and columns can only be (1, 2, 3)')
            continue
        if current_state[row-1][column-1]=='-':
            current_state[row-1][column-1]='o'
            break
    return current_state

In [18]:
def tic_tac_toe(board):
    current_state=list([list(row[:]) for row in board])
    states_points=get_states_values(board=current_state)
    while True:
        if check_win(board=current_state) is not None:
            print('Computer Won!') if check_win(board=current_state)=='x' else print('You Won!')
            break
        if check_draw(board=current_state):
            print('Match Drawn!')
            break
        current_player=get_player_turn(board=current_state)
        if current_player=='x':
            current_state=computer_turn(board=current_state, states_points=states_points)
        else:
            current_state=human_turn(board=current_state)
        print('After move:')
        display_board(board=current_state)

In [19]:
def main():
    board=initialize_board()
    board=setup_board(board=board, random_count=0)
    display_board(board=board)
    tic_tac_toe(board=board)
if __name__=='__main__':
    main()

['-', '-', '-']
['-', '-', '-']
['-', '-', '-']
After move:
['x', '-', '-']
['-', '-', '-']
['-', '-', '-']
Enter a valid cell to place your mark!
After move:
['x', '-', '-']
['-', 'o', '-']
['-', '-', '-']
After move:
['x', 'x', '-']
['-', 'o', '-']
['-', '-', '-']
Enter a valid cell to place your mark!
After move:
['x', 'x', 'o']
['-', 'o', '-']
['-', '-', '-']
After move:
['x', 'x', 'o']
['-', 'o', '-']
['x', '-', '-']
Enter a valid cell to place your mark!
After move:
['x', 'x', 'o']
['o', 'o', '-']
['x', '-', '-']
After move:
['x', 'x', 'o']
['o', 'o', 'x']
['x', '-', '-']
Enter a valid cell to place your mark!
After move:
['x', 'x', 'o']
['o', 'o', 'x']
['x', '-', 'o']
After move:
['x', 'x', 'o']
['o', 'o', 'x']
['x', 'x', 'o']
Match Drawn!


In [22]:
def test():
    board=[
        ['x', 'o', '-'],
        ['x', 'o', 'o'],
        ['-', '-', 'x']
    ]
    children=get_child_states(board=board)
    print(board)
    print(children)
    states_points=dict()
    print(get_state_value(board=board, states_points=states_points))
    states_points_keys=list(states_points.keys())
    for key in states_points_keys:
        display_board(key)
        print(states_points[key])
test()

[['x', 'o', '-'], ['x', 'o', 'o'], ['-', '-', 'x']]
[[['x', 'o', 'x'], ['x', 'o', 'o'], ['-', '-', 'x']], [['x', 'o', '-'], ['x', 'o', 'o'], ['x', '-', 'x']], [['x', 'o', '-'], ['x', 'o', 'o'], ['-', 'x', 'x']]]
1
('x', 'o', 'x')
('x', 'o', 'o')
('o', 'x', 'x')
0
('x', 'o', 'x')
('x', 'o', 'o')
('o', '-', 'x')
0
('x', 'o', 'x')
('x', 'o', 'o')
('-', 'o', 'x')
-1
('x', 'o', 'x')
('x', 'o', 'o')
('-', '-', 'x')
-1
('x', 'o', '-')
('x', 'o', 'o')
('x', '-', 'x')
1
('x', 'o', 'o')
('x', 'o', 'o')
('x', 'x', 'x')
1
('x', 'o', 'o')
('x', 'o', 'o')
('-', 'x', 'x')
1
('x', 'o', '-')
('x', 'o', 'o')
('o', 'x', 'x')
0
('x', 'o', '-')
('x', 'o', 'o')
('-', 'x', 'x')
0
('x', 'o', '-')
('x', 'o', 'o')
('-', '-', 'x')
1
